## 02. Feature Engineering

`Topollo Naketsana`

### This notebook aims to build customer-level features, their 'ances and transactions, as we have explored the data behavior from the EDA stage




In [101]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

### Load the data

In [102]:
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')

txn = pd.read_parquet('transactions_features.parquet')
fin = pd.read_parquet('financials_features.parquet')
demo = pd.read_parquet('demographics_clean.parquet')

In [103]:
train.shape

(8360, 2)

In [104]:
txn.info()

<class 'pandas.DataFrame'>
RangeIndex: 18017073 entries, 0 to 18017072
Data columns (total 9 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   UniqueID                     string        
 1   AccountID                    string        
 2   TransactionDate              datetime64[ns]
 3   TransactionAmount            float64       
 4   TransactionTypeDescription   str           
 5   TransactionBatchDescription  str           
 6   StatementBalance             float64       
 7   IsDebitCredit                str           
 8   ReversalTypeDescription      str           
dtypes: datetime64[ns](1), float64(2), str(4), string(2)
memory usage: 3.2 GB


###

### 2.1. Transaction features

In [105]:
# the transact features
txn_features = txn.groupby('UniqueID').agg({
    'TransactionAmount': [
        'sum', 'mean', 'median', 'std',
        'count', 'max', 'min'
    ],
    'StatementBalance': [
        'mean', 'median', 'std',
        'min', 'max'
    ]
})

In [106]:
# flatten the txn features
txn_features.columns = [
    '_'.join(col)
    for col in txn_features.columns
]

txn_features = txn_features.reset_index()

In [107]:
txn_features.head()

,UniqueID,TransactionAmount_sum,TransactionAmount_mean,TransactionAmount_median,TransactionAmount_std,TransactionAmount_count,TransactionAmount_max,TransactionAmount_min,StatementBalance_mean,StatementBalance_median,StatementBalance_std,StatementBalance_min,StatementBalance_max
0,00038178-a14e-468a-8053-a0edf5cf1b02,-10877.41,-3.943949,-157.295,4487.673801,2758,139747.29,-27975.48,-2396.908434,-10236.505,26412.679422,-44656.79,158001.73
1,0006aac7-06d3-4fee-b0f6-11f57b9419e9,132923.43,196.341846,-351.480,14810.330258,677,244521.99,-81712.49,42721.482541,32915.050,38020.872991,-1619.16,314477.68
2,00093e2d-9e1e-4061-ad27-a79b8ff9e165,-894563.40,-572.703841,-925.520,88833.409077,1562,1000000.00,-1000000.00,79175.340800,39337.810,152572.842104,-44149.01,1000000.00
3,0011d60f-a4e2-4333-81fc-2d557a82109b,-2000656.68,-7970.743745,-301.590,132902.058472,251,1000000.00,-1000000.00,463595.304900,177155.140,436019.174486,35905.31,1000000.00
4,0016f1e2-64c1-4c65-a668-1dc6bf3b5875,-127.16,-2.890000,-2.020,3.039344,44,0.00,-8.05,279.836136,287.955,48.508435,184.19,369.23


In [108]:
# add the credit or debit column to the table
debit_credit = pd.crosstab(
    txn['UniqueID'],
    txn['IsDebitCredit']
).reset_index()

txn_features = txn_features.merge(
    debit_credit,
    on= 'UniqueID',
    how= 'left'
)

In [109]:
txn_features.head()

,UniqueID,TransactionAmount_sum,TransactionAmount_mean,TransactionAmount_median,TransactionAmount_std,TransactionAmount_count,TransactionAmount_max,TransactionAmount_min,StatementBalance_mean,StatementBalance_median,StatementBalance_std,StatementBalance_min,StatementBalance_max,Credit,Debit
0,00038178-a14e-468a-8053-a0edf5cf1b02,-10877.41,-3.943949,-157.295,4487.673801,2758,139747.29,-27975.48,-2396.908434,-10236.505,26412.679422,-44656.79,158001.73,2575,183
1,0006aac7-06d3-4fee-b0f6-11f57b9419e9,132923.43,196.341846,-351.480,14810.330258,677,244521.99,-81712.49,42721.482541,32915.050,38020.872991,-1619.16,314477.68,533,144
2,00093e2d-9e1e-4061-ad27-a79b8ff9e165,-894563.40,-572.703841,-925.520,88833.409077,1562,1000000.00,-1000000.00,79175.340800,39337.810,152572.842104,-44149.01,1000000.00,1351,211
3,0011d60f-a4e2-4333-81fc-2d557a82109b,-2000656.68,-7970.743745,-301.590,132902.058472,251,1000000.00,-1000000.00,463595.304900,177155.140,436019.174486,35905.31,1000000.00,174,77
4,0016f1e2-64c1-4c65-a668-1dc6bf3b5875,-127.16,-2.890000,-2.020,3.039344,44,0.00,-8.05,279.836136,287.955,48.508435,184.19,369.23,22,22


###

### 2.2. Financial features

In [110]:
fin_features = fin.groupby('UniqueID').agg({
    'NetInterestIncome': ['sum', 'mean', 'std'],
    'NetInterestRevenue': ['sum', 'mean'],
    'Product': 'nunique'
})

In [111]:
# flattened
fin_features.columns = [
    '_'.join(col) if isinstance(col, tuple) else col
    for col in fin_features.columns
]

fin_features = fin_features.reset_index()

In [112]:
# financial columns
financial_cols = [
    'NetInterestIncome_sum',
    'NetInterestIncome_mean',
    'NetInterestIncome_std',
    'NetInterestRevenue_sum',
    'NetInterestRevenue_mean',
    'Product_nunique'
]

###

### 2.3. Democratic features

In [113]:
demo_features = demo[[
    'UniqueID',
    'Gender',
    'IncomeCategory',
    'AnnualGrossIncome',
    'OccupationCategory',
    'CustomerStatus'
]]

### 2.4. Combine datasents

In [114]:
train_df = train.merge(txn_features, on= 'UniqueID', how= 'left')
train_df = train_df.merge(fin_features, on= 'UniqueID', how= 'left').drop(columns= ['index'], errors= 'ignore')
train_df = train_df.merge(demo_features, on= 'UniqueID', how= 'left')

test_df = test.merge(txn_features, on= 'UniqueID', how= 'left')
test_df = test_df.merge(fin_features, on= 'UniqueID', how= 'left').drop(columns= ['index'], errors= 'ignore')
test_df = test_df.merge(demo_features, on= 'UniqueID', how= 'left')


# fill in the missing values
# numeric fill
train_df['TransactionAmount_std'] = train_df['TransactionAmount_std'].fillna(0)
train_df['StatementBalance_std'] = train_df['StatementBalance_std'].fillna(0)

test_df['TransactionAmount_std'] = test_df['TransactionAmount_std'].fillna(0)
test_df['StatementBalance_std'] = test_df['StatementBalance_std'].fillna(0)

train_df[financial_cols] = train_df[financial_cols].fillna(0)
test_df[financial_cols] = test_df[financial_cols].fillna(0)

# gender 
train_df['Gender'] = train_df['Gender'].fillna('Unknown')
test_df['Gender'] = test_df['Gender'].fillna('Unknown')

# annual income - fill with median
train_df['AnnualGrossIncome'] = train_df['AnnualGrossIncome'].fillna(
    train_df['AnnualGrossIncome'].median()
)

test_df['AnnualGrossIncome'] = test_df['AnnualGrossIncome'].fillna(
    test_df['AnnualGrossIncome'].median()
)

In [115]:
train_df.head()

,UniqueID,next_3m_txn_count,TransactionAmount_sum,TransactionAmount_mean,TransactionAmount_median,TransactionAmount_std,TransactionAmount_count,TransactionAmount_max,TransactionAmount_min,StatementBalance_mean,...,NetInterestIncome_mean,NetInterestIncome_std,NetInterestRevenue_sum,NetInterestRevenue_mean,Product_nunique,Gender,IncomeCategory,AnnualGrossIncome,OccupationCategory,CustomerStatus
0,00093e2d-9e1e-4061-ad27-a79b8ff9e165,129,-894563.40,-572.703841,-925.520,88833.409077,1562,1000000.00,-1000000.00,79175.340800,...,1304.165333,2414.125438,10647.86,236.619111,3.0,M,High Income,2244579.16,Management / Executive,Active Customer
1,0011d60f-a4e2-4333-81fc-2d557a82109b,16,-2000656.68,-7970.743745,-301.590,132902.058472,251,1000000.00,-1000000.00,463595.304900,...,-1775.770476,3450.277593,20482.99,975.380476,1.0,M,High Income,1251211.80,Management / Executive,Active Customer
2,0016f1e2-64c1-4c65-a668-1dc6bf3b5875,117,-127.16,-2.890000,-2.020,3.039344,44,0.00,-8.05,279.836136,...,-2.015238,0.428388,105.33,5.015714,1.0,M,Middle Income,344297.42,Other / Unclassified,Active Customer
3,001aa3c5-632d-435e-a421-cc3615ccef4d,70,87729.20,258.027059,-895.765,89647.208240,340,1000000.00,-698552.99,81728.208706,...,-433.280976,852.424778,6773.93,165.217805,2.0,F,Very High Income,1846455.79,Management / Executive,Active Customer
4,00298c6f-4f9d-4f28-b72c-ad0e56e9eb84,393,44110.45,20.650960,-143.645,9755.953589,2136,122286.62,-38055.67,155872.668043,...,-463.207273,859.880422,6383.57,290.162273,2.0,M,Very High Income,2176785.64,Management / Executive,Active Customer


In [116]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8360 entries, 0 to 8359
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   UniqueID                  8360 non-null   object 
 1   next_3m_txn_count         8360 non-null   int64  
 2   TransactionAmount_sum     8360 non-null   float64
 3   TransactionAmount_mean    8360 non-null   float64
 4   TransactionAmount_median  8360 non-null   float64
 5   TransactionAmount_std     8360 non-null   float64
 6   TransactionAmount_count   8360 non-null   int64  
 7   TransactionAmount_max     8360 non-null   float64
 8   TransactionAmount_min     8360 non-null   float64
 9   StatementBalance_mean     8360 non-null   float64
 10  StatementBalance_median   8360 non-null   float64
 11  StatementBalance_std      8360 non-null   float64
 12  StatementBalance_min      8360 non-null   float64
 13  StatementBalance_max      8360 non-null   float64
 14  Credit             

In [117]:
train_df.isna().sum()

UniqueID                    0
next_3m_txn_count           0
TransactionAmount_sum       0
TransactionAmount_mean      0
TransactionAmount_median    0
TransactionAmount_std       0
TransactionAmount_count     0
TransactionAmount_max       0
TransactionAmount_min       0
StatementBalance_mean       0
StatementBalance_median     0
StatementBalance_std        0
StatementBalance_min        0
StatementBalance_max        0
Credit                      0
Debit                       0
NetInterestIncome_sum       0
NetInterestIncome_mean      0
NetInterestIncome_std       0
NetInterestRevenue_sum      0
NetInterestRevenue_mean     0
Product_nunique             0
Gender                      0
IncomeCategory              0
AnnualGrossIncome           0
OccupationCategory          0
CustomerStatus              0
dtype: int64

In [118]:
print(f'shape: {train_df.shape}')
print(f'shape: {test_df.shape}')

shape: (8360, 27)
shape: (3584, 26)


In [119]:
# save
train_df.to_csv('train_engineered.csv', index= False)
test_df.to_csv('test_engineered.csv', index= False)